In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import numpy.linalg as LA
import pandas as pd
import jax
import jax.numpy as jnp
from jax.typing import ArrayLike
import numpyro
import numpyro.distributions as dist
from numpyro.contrib.hsgp.approximation import hsgp_matern

jax.config.update("jax_enable_x64", True)

import altair as alt

from principled_pclm.sim import PoissonEnveloped1D, make_aggregation_matrix
from principled_pclm.models._inference import run_inference_mcmc

from principled_pclm.models import (
    expand_interval_dataframe,
    compute_interval_widths,
    build_composition_matrix,
)

## Gaussian Process Basics

Before fitting the HSGP model below, it's useful to review the spectral density $S(\omega)$ associated with a stationary covariance kernel $k(r)$, $r = x - x'$. By Bochner's theorem, $k$ and $S$ form a Fourier transform pair:

$$S(\omega) = \int_{-\infty}^{\infty} k(r)\, e^{-i\omega r}\, dr$$

The Hilbert-space GP (HSGP) approximation used below replaces the exact kernel with a finite basis expansion whose coefficients are weighted by $S(\omega)$ evaluated at the Laplacian eigenfrequencies, so an explicit spectral density is required for every kernel we want to approximate. Below we implement the 1-D spectral densities for the squared exponential kernel and the Matérn kernels with $\nu = 3/2$ and $\nu = 5/2$, following Rasmussen & Williams (2006, Eq. 4.15) and Riutort-Mayol et al. (2023).

In [3]:
def spectral_density_squared_exponential(
    omega: ArrayLike, alpha: float, length: float
) -> ArrayLike:
    r"""Spectral density of the 1-D squared exponential kernel.

    .. math::
        S(\omega) = \alpha \sqrt{2\pi}\,\ell\,
            \exp\left(-\tfrac{1}{2}\ell^2\omega^2\right)

    :param omega: angular frequency at which to evaluate the spectral density
    :param alpha: kernel amplitude (variance), i.e. k(0)
    :param length: kernel length scale
    :return: spectral density evaluated at ``omega``
    """
    omega = jnp.asarray(omega)
    return alpha * jnp.sqrt(2 * jnp.pi) * length * jnp.exp(-0.5 * length**2 * omega**2)


def spectral_density_matern32(
    omega: ArrayLike, alpha: float, length: float
) -> ArrayLike:
    r"""Spectral density of the 1-D Matérn kernel with :math:`\nu = 3/2`.

    .. math::
        S(\omega) = \alpha\, \frac{4\lambda^3}{(\lambda^2 + \omega^2)^2},
        \qquad \lambda = \frac{\sqrt{3}}{\ell}

    :param omega: angular frequency at which to evaluate the spectral density
    :param alpha: kernel amplitude (variance), i.e. k(0)
    :param length: kernel length scale
    :return: spectral density evaluated at ``omega``
    """
    omega = jnp.asarray(omega)
    lam = jnp.sqrt(3.0) / length
    return alpha * 4 * lam**3 / (lam**2 + omega**2) ** 2


def spectral_density_matern52(
    omega: ArrayLike, alpha: float, length: float
) -> ArrayLike:
    r"""Spectral density of the 1-D Matérn kernel with :math:`\nu = 5/2`.

    .. math::
        S(\omega) = \alpha\, \frac{16\lambda^5}{3(\lambda^2 + \omega^2)^3},
        \qquad \lambda = \frac{\sqrt{5}}{\ell}

    :param omega: angular frequency at which to evaluate the spectral density
    :param alpha: kernel amplitude (variance), i.e. k(0)
    :param length: kernel length scale
    :return: spectral density evaluated at ``omega``
    """
    omega = jnp.asarray(omega)
    lam = jnp.sqrt(5.0) / length
    return alpha * (16.0 / 3.0) * lam**5 / (lam**2 + omega**2) ** 3

In [4]:
ALPHA = 0.5
LENGTHSCALE = 3.051
omega = jnp.linspace(0, 5, 500)

spd_se = spectral_density_squared_exponential(
    omega=omega, alpha=ALPHA, length=LENGTHSCALE
)
spd_m32 = spectral_density_matern32(omega=omega, alpha=ALPHA, length=LENGTHSCALE)
spd_m52 = spectral_density_matern52(omega=omega, alpha=ALPHA, length=LENGTHSCALE)

df_spd = pd.DataFrame(
    {
        "omega": np.tile(omega, 3),
        "spd": np.concatenate([spd_se, spd_m32, spd_m52]),
        "kernel": np.repeat(["SE", "Matern 3/2", "Matern 5/2"], len(omega)),
    }
)

chart_spd = (
    alt.Chart(df_spd)
    .mark_line()
    .encode(
        x=alt.X("omega", title="Angular Frequency"),
        y=alt.Y("spd", title="Spectral Density"),
        color=alt.Color("kernel", title="Kernel"),
    )
    .properties(title="Spectral Density of the Kernels")
)

### Sampling from a GP Prior

We can also sample functions directly from a GP prior by building the kernel's Gram matrix $K_{ij} = k(x_i, x_j)$ on a finite grid $x$ and drawing $f \sim \mathcal{N}(0, K)$ via a Cholesky factorization $K = LL^\top$, $f = Lz$ with $z \sim \mathcal{N}(0, I)$. This gives us a reference (exact, $O(n^3)$) sampler to compare against the HSGP approximation used later in this notebook.

In [5]:
def kernel_squared_exponential(
    x1: ArrayLike, x2: ArrayLike, alpha: float, length: float
) -> ArrayLike:
    r"""Squared exponential covariance matrix between two 1-D input sets.

    .. math::
        k(x, x') = \alpha \exp\left(-\frac{(x - x')^2}{2\ell^2}\right)
    """
    x1, x2 = jnp.asarray(x1), jnp.asarray(x2)
    r2 = (x1[:, None] - x2[None, :]) ** 2
    return alpha * jnp.exp(-0.5 * r2 / length**2)


def kernel_matern32(
    x1: ArrayLike, x2: ArrayLike, alpha: float, length: float
) -> ArrayLike:
    r"""Matérn :math:`\nu = 3/2` covariance matrix between two 1-D input sets.

    .. math::
        k(x, x') = \alpha \left(1 + \sqrt{3}\,\tfrac{|x - x'|}{\ell}\right)
            \exp\left(-\sqrt{3}\,\tfrac{|x - x'|}{\ell}\right)
    """
    x1, x2 = jnp.asarray(x1), jnp.asarray(x2)
    r = jnp.abs(x1[:, None] - x2[None, :])
    scaled = jnp.sqrt(3.0) * r / length
    return alpha * (1 + scaled) * jnp.exp(-scaled)


def kernel_matern52(
    x1: ArrayLike, x2: ArrayLike, alpha: float, length: float
) -> ArrayLike:
    r"""Matérn :math:`\nu = 5/2` covariance matrix between two 1-D input sets.

    .. math::
        k(x, x') = \alpha \left(1 + \sqrt{5}\,\tfrac{|x - x'|}{\ell}
            + \tfrac{5}{3}\tfrac{(x - x')^2}{\ell^2}\right)
            \exp\left(-\sqrt{5}\,\tfrac{|x - x'|}{\ell}\right)
    """
    x1, x2 = jnp.asarray(x1), jnp.asarray(x2)
    r = jnp.abs(x1[:, None] - x2[None, :])
    scaled = jnp.sqrt(5.0) * r / length
    return alpha * (1 + scaled + scaled**2 / 3) * jnp.exp(-scaled)


def sample_gp(
    key: jax.Array,
    x: ArrayLike,
    kernel_fn,
    alpha: float,
    length: float,
    mean: ArrayLike = 0.0,
    jitter: float = 1e-6,
    num_samples: int = 1,
) -> ArrayLike:
    """Draw samples from a zero-mean GP prior with the given kernel on a fixed grid.

    :param key: JAX PRNG key
    :param x: 1-D array of input locations
    :param kernel_fn: one of ``kernel_squared_exponential``, ``kernel_matern32``,
        ``kernel_matern52`` (or any callable with the same signature)
    :param alpha: kernel amplitude (variance), i.e. k(0)
    :param length: kernel length scale
    :param mean: prior mean, broadcastable to ``x.shape``
    :param jitter: value added to the diagonal for numerical stability
    :param num_samples: number of independent draws
    :return: array of shape ``(num_samples, len(x))``
    """
    x = jnp.asarray(x)
    n = x.shape[0]
    cov = kernel_fn(x, x, alpha, length) + jitter * jnp.eye(n)
    L = jnp.linalg.cholesky(cov)
    z = jax.random.normal(key, (n, num_samples))
    return (mean + L @ z).T


def sample_gp_squared_exponential(
    key: jax.Array,
    x: ArrayLike,
    alpha: float,
    length: float,
    mean: ArrayLike = 0.0,
    jitter: float = 1e-6,
    num_samples: int = 1,
) -> ArrayLike:
    """Draw samples from a GP prior with the squared exponential kernel."""
    return sample_gp(
        key,
        x,
        kernel_squared_exponential,
        alpha,
        length,
        mean=mean,
        jitter=jitter,
        num_samples=num_samples,
    )


def sample_gp_matern32(
    key: jax.Array,
    x: ArrayLike,
    alpha: float,
    length: float,
    mean: ArrayLike = 0.0,
    jitter: float = 1e-6,
    num_samples: int = 1,
) -> ArrayLike:
    r"""Draw samples from a GP prior with the Matérn :math:`\nu = 3/2` kernel."""
    return sample_gp(
        key,
        x,
        kernel_matern32,
        alpha,
        length,
        mean=mean,
        jitter=jitter,
        num_samples=num_samples,
    )


def sample_gp_matern52(
    key: jax.Array,
    x: ArrayLike,
    alpha: float,
    length: float,
    mean: ArrayLike = 0.0,
    jitter: float = 1e-6,
    num_samples: int = 1,
) -> ArrayLike:
    r"""Draw samples from a GP prior with the Matérn :math:`\nu = 5/2` kernel."""
    return sample_gp(
        key,
        x,
        kernel_matern52,
        alpha,
        length,
        mean=mean,
        jitter=jitter,
        num_samples=num_samples,
    )

In [6]:
ALPHA = 0.5
LENGTHSCALE = 3.051
x = jnp.linspace(0, 10, 100)

sample_gp_se = sample_gp_squared_exponential(
    key=jax.random.PRNGKey(0),
    x=x,
    alpha=ALPHA,
    length=LENGTHSCALE,
    num_samples=1,
).flatten()

sample_gp_m32 = sample_gp_matern32(
    key=jax.random.PRNGKey(1),
    x=x,
    alpha=ALPHA,
    length=LENGTHSCALE,
    num_samples=1,
).flatten()

sample_gp_m52 = sample_gp_matern52(
    key=jax.random.PRNGKey(2),
    x=x,
    alpha=ALPHA,
    length=LENGTHSCALE,
    num_samples=1,
).flatten()

df_sample = pd.DataFrame(
    {
        "x": np.tile(x, 3),
        "y": np.concatenate([sample_gp_se, sample_gp_m32, sample_gp_m52]),
        "kernel": np.repeat(["SE", "Matern 3/2", "Matern 5/2"], len(x)),
    }
)

chart_sample = (
    alt.Chart(df_sample)
    .mark_line()
    .encode(
        x=alt.X("x", title="x"),
        y=alt.Y("y", title="y"),
        color=alt.Color("kernel", title="Kernel"),
    )
    .properties(title="Samples from GP Priors with Different Kernels")
)

In [7]:
alt.hconcat(chart_spd, chart_sample).resolve_scale(y="independent").display()

alt.HConcatChart(...)

## Experiment Setup

In [26]:
# Generate the dataset
# left_limits = [0, 0, 5, 8, 10, 15, 18, 20, 25, 30, 35, 40, 50, 60, 70, 80, 90]
# right_limits = [5, 12, 15, 25, 20, 30, 35, 40, 45, 50, 55, 60, 70, 80, 90, 95, 99]

# Mixed size intervals
left_limits = [0, 10, 20, 30, 50, 70, 80, 90]
right_limits = [9, 19, 29, 49, 69, 79, 89, 99]

# Size 10 intervals
# left_limits = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
# right_limits = [9, 19, 29, 39, 49, 59, 69, 79, 89, 99]

# Size 20 intervals
# left_limits = [0, 20, 40, 60, 80]
# right_limits = [19, 39, 59, 79, 99]

intervals = [
    f"[{left},{right}{']' if right in {5, 15, 25, 30, 40, 50, 60, 70, 80, 90, 99} else ')'}"
    for left, right in zip(left_limits, right_limits)
]
print("Intervals:", intervals)

df_obs, df_rates = PoissonEnveloped1D(
    obs_intervals=intervals, n_observations=100, amplitude_decay=1.0, seed=42
).generate(return_true_rates=True)

Intervals: ['[0,9)', '[10,19)', '[20,29)', '[30,49)', '[50,69)', '[70,79)', '[80,89)', '[90,99]']


In [27]:
def summarize_interval_segments(df_segment: pd.DataFrame) -> pd.DataFrame:
    return (
        df_segment[["obs_interval", "x", "lambda"]]
        .drop_duplicates(["obs_interval", "x"])
        .groupby("obs_interval", as_index=False)
        .agg(x_min=("x", "min"), x_max=("x", "max"), lambda_obs=("lambda", "mean"))
    )


def plot_data(df_obs: pd.DataFrame, df_rates: pd.DataFrame) -> alt.Chart:
    df_segment = expand_interval_dataframe(
        df_obs, interval_col="obs_interval", x_col="x"
    )
    source_segment = summarize_interval_segments(df_segment)

    df_emp = df_obs.groupby("obs_interval")["y"].agg(["sum", "count"]).reset_index()
    df_emp["lambda"] = df_emp["sum"] / df_emp["count"]
    df_emp = expand_interval_dataframe(df_emp, interval_col="obs_interval", x_col="x")
    source_emp = summarize_interval_segments(df_emp)

    chart_true = (
        alt.Chart(df_rates)
        .mark_line(color="red")
        .encode(x="x", y=alt.Y("lambda:Q", title="lambda"))
    )

    chart_segment = (
        alt.Chart(source_segment)
        .mark_rule(color="orange", strokeWidth=2.5, opacity=0.85)
        .encode(
            x=alt.X("x_min:Q"),
            x2="x_max:Q",
            y=alt.Y("lambda_obs:Q", title="lambda"),
        )
    )

    chart_emp = (
        alt.Chart(source_emp)
        .mark_rule(color="blue", strokeWidth=2.5, opacity=0.85)
        .encode(
            x=alt.X("x_min:Q"),
            x2="x_max:Q",
            y=alt.Y("lambda_obs:Q", title="lambda"),
        )
    )

    return chart_true + chart_segment + chart_emp

In [28]:
plot_data(df_obs, df_rates)

alt.LayerChart(...)

In [11]:
def summarize_posterior_lam(
    mcmc_samples: dict, x: np.ndarray, label: str
) -> pd.DataFrame:
    """Posterior mean and 95% band for `lam`, tagged with an experiment label."""
    central = mcmc_samples["lam"].mean(axis=0)
    lower = np.percentile(mcmc_samples["lam"], 2.5, axis=0)
    upper = np.percentile(mcmc_samples["lam"], 97.5, axis=0)
    return pd.DataFrame(
        {
            "x": x,
            "lam": central.flatten(),
            "lower": lower.flatten(),
            "upper": upper.flatten(),
            "experiment": label,
        }
    )


def plot_posterior_band(
    source: pd.DataFrame,
    df_obs: pd.DataFrame,
    df_rates: pd.DataFrame,
    title: str | None = None,
) -> alt.Chart:
    """Posterior mean + 95% band, layered over the true curve and raw interval markers."""
    chart = alt.Chart(source).mark_line(color="purple").encode(x="x", y="lam")
    band = (
        alt.Chart(source)
        .mark_area(color="purple", opacity=0.2)
        .encode(x="x", y="lower", y2="upper")
    )
    result = plot_data(df_obs, df_rates) + chart + band
    return result.properties(title=title) if title else result

## Modeling

In [12]:
# Prepare data for inference
df_model = df_obs.groupby(["obs_interval_id"])["y"].agg(N="size", y="sum").reset_index()

# Convert to JAX arrays
x = jnp.arange(0, 100)
y = jnp.array(df_model["y"].values)
interval_id = jnp.array(df_model["obs_interval_id"].values)
N = jnp.array(df_model["N"].values)
C = jnp.array(make_aggregation_matrix(left_limits, right_limits))

In [13]:
class NaiveHSGP:
    """
    HSGP-based Bayesian regression model for aggregated Poisson counts,
    using a Matern 3/2 kernel with a fixed lengthscale.
    """

    def __init__(
        self,
        x: ArrayLike,
        C: jnp.ndarray,
        N: ArrayLike,
        interval_id: ArrayLike,
        length: float,
        ell: float | None = None,
        m: int | None = None,
    ):
        """
        Parameters
        ----------
        x : ArrayLike
            Fine-scale grid locations at which the latent log-rate is evaluated.
            Centered automatically before being passed to the HSGP approximation.
        C : ArrayLike
            Aggregation matrix mapping fine-scale rates to observed intervals.
        N : ArrayLike
            Number of observations in each interval.
        interval_id : ArrayLike
            Interval indices for each observation.
        length : float
            Fixed lengthscale of the Matern 3/2 kernel.
        ell : float, optional
            Boundary factor for the HSGP approximation; the approximation is
            valid on the interval [-ell, ell]. If None, defaults to 1.5 times
            the largest absolute centered grid location, a standard rule of
            thumb (Riutort-Mayol et al., 2023) that extends the domain far
            enough past the data to control boundary effects.
        m : int, optional
            Number of basis functions used in the HSGP approximation. If
            None, defaults to a rule of thumb that scales with `ell / length`
            (shorter lengthscales need higher-frequency basis functions, and
            therefore more of them). For critical applications, check this
            default by comparing against the exact GP sampler defined above
            (e.g. ``sample_gp_matern32``) or by increasing `m`/`ell` until
            results stop changing.
        """
        self.x = jnp.asarray(x)
        self.xc = self.x - jnp.mean(self.x)
        self.C = C
        self.N = N
        self.interval_id = interval_id
        self.length = length

        self.ell = (
            float(ell) if ell is not None else float(1.5 * jnp.max(jnp.abs(self.xc)))
        )
        self.m = (
            int(m)
            if m is not None
            else int(max(jnp.ceil(4.5 * self.ell / self.length), 10))
        )

    def model(self, y=None):
        W = self.C.sum(axis=1)

        # --- Priors ---
        beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
        alpha = numpyro.sample("alpha", dist.LogNormal(0, 1))

        # --- HSGP approximation of the latent GP (Matern nu=3/2, fixed lengthscale) ---
        f = hsgp_matern(
            x=self.xc,
            nu=3 / 2,
            alpha=alpha,
            length=self.length,
            ell=self.ell,
            m=self.m,
            non_centered=True,
        )

        # --- Latent rate ---
        lam = numpyro.deterministic("lam", jnp.exp(beta0 + f))
        lam_interval = jnp.matmul(self.C, lam) / W * self.N[self.interval_id]

        # --- Likelihood ---
        with numpyro.plate("obs", len(y)):
            numpyro.sample("y", dist.Poisson(lam_interval[self.interval_id]), obs=y)

In [14]:
# Baseline model with naive lengthscale selection
naive_hsgp_bl = NaiveHSGP(
    x=x,
    C=C,
    N=N,
    interval_id=interval_id,
    ell=2.0 * jnp.max(jnp.abs(x - jnp.mean(x))),
    length=1.0,
)

In [15]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    naive_hsgp_bl.model,
    y=y,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:03<00:00, 293.48it/s, 127 steps of size 3.22e-02. acc. prob=0.95]


Number of divergences: 0
dict_keys(['alpha', 'beta', 'beta0', 'lam'])


In [16]:
source_exp1 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "1. Naive"
)
plot_posterior_band(source_exp1, df_obs, df_rates)

alt.LayerChart(...)

In [23]:
# Baseline model with naive lengthscale selection
naive_hsgp_trunc = NaiveHSGP(
    x=x,
    C=C,
    N=N,
    interval_id=interval_id,
    ell=2.0 * jnp.max(jnp.abs(x - jnp.mean(x))),
    length=5 * 3.051,
)

In [24]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    naive_hsgp_trunc.model,
    y=y,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 1050.06it/s, 127 steps of size 4.31e-02. acc. prob=0.91]


Number of divergences: 0
dict_keys(['alpha', 'beta', 'beta0', 'lam'])


In [25]:
source_exp1 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "1. Naive"
)
plot_posterior_band(source_exp1, df_obs, df_rates)

alt.LayerChart(...)